# QLoRA Demo

In this tutorial, we use `v5e-8` TPU to demonstrate how to fine-tune the [Gemma](https://deepmind.google/models/gemma/)
model for translation using [Low Rank Adaptation(LoRA)](https://arxiv.org/abs/2106.09685), a
parameter-efficient way of finetuning LLMs.

LoRA works by freezing the original weights of the pre-trained model and
injecting trainable low-rank matrices into each layer of the Transformer
architecture. During fine-tuning, only these newly introduced low-rank matrices
are updated, greatly decreasing the computational and memory resources required
compared to traditional full fine-tuning. This approach is based on the
observation that the changes in model weights needed for adaptation often have a
low rank. The benefits of using LoRA include reduced HBM memory usage, faster
training times, and the advantage that, after training, the LoRA adapters can be
merged with the original model weights, resulting in no additional inference
latency.

## Install necessary libraries

In [ ]:
import os
import sys

module_path = os.path.dirname(os.path.abspath(os.curdir))
if module_path not in sys.path:
  sys.path.insert(0, module_path)

In [ ]:
import gc
import logging

import dotenv
from flax import nnx
import jax
import jax.numpy as jnp
import optax
from orbax import checkpoint as ocp
import qwix
from tunix.examples.data import translation_dataset as data_lib
from tunix.generate import sampler as sampler_lib
from tunix.generate import tokenizer_adapter as tokenizer_lib
from tunix.models.gemma import model as gemma_lib
from tunix.models.gemma import params as params_lib
from tunix.sft import metrics_logger
from tunix.sft import peft_trainer
from tunix.sft import utils

logger = logging.getLogger()
logger.setLevel(logging.INFO)

## Hyperparameters

In [ ]:
# Data
BATCH_SIZE = 16  # Adjust based on your TPU memory and model size.
MAX_TARGET_LENGTH = 256

# Model
MESH = [
    (1, 8),
    ("fsdp", "tp"),
]  # Adjust based on your TPU memory and model size.
# LoRA
RANK = 16
ALPHA = 2.0

# Train
MAX_STEPS = 100
EVAL_EVERY_N_STEPS = 20
NUM_EPOCHS = 3


# Checkpoint saving
INTERMEDIATE_CKPT_DIR = "/tmp/content/intermediate_ckpt/"
CKPT_DIR = "/tmp/content/ckpts/"
PROFILING_DIR = "/tmp/content/profiling/"

In [ ]:
def create_dir(path):
  try:
    os.makedirs(path, exist_ok=True)
    logging.info(f"Created dir: {path}")
  except OSError as e:
    logging.error(f"Error creating directory '{path}': {e}")


create_dir(INTERMEDIATE_CKPT_DIR)
create_dir(CKPT_DIR)
create_dir(PROFILING_DIR)

## Load model from Kaggle

To load the model, you need to be on [Kaggle](https://www.kaggle.com/) and need
to have agreed to the Gemma license
[here](https://www.kaggle.com/models/google/gemma/flax/).

In [ ]:
import os
import kagglehub

# Log in
if "KAGGLE_USERNAME" not in os.environ or "KAGGLE_KEY" not in os.environ:
  kagglehub.login()

# alternatively place kaggle.json under ~/.kaggle/

In [ ]:
model_path = {
    "gemma": "google/gemma/flax/",
    "qwen3": "qwen-lm/qwen-3/transformers/",
}
model_family = "gemma"
model_version = "2b"
kaggle_ckpt_path = kagglehub.model_download(
    f"{model_path[model_family]}{model_version}"
)

In [ ]:
def get_base_model(ckpt_path, model_version):
  if model_version == '2b':
    model_config = gemma_lib.ModelConfig.gemma_2b()
  elif model_version == '7b':
    model_config = gemma_lib.ModelConfig.gemma_7b()
  else:
    raise ValueError(f'Unsupported model version {model_version}')
  mesh = jax.make_mesh(*MESH)
  abs_gemma: nnx.Module = nnx.eval_shape(
      lambda: gemma_lib.Transformer(model_config, rngs=nnx.Rngs(params=0))
  )
  abs_state = nnx.state(abs_gemma)
  abs_state = jax.tree.map(
      lambda a, s: jax.ShapeDtypeStruct(a.shape, jnp.bfloat16, sharding=s),
      abs_state,
      nnx.get_named_sharding(abs_state, mesh),
  )
  checkpointer = ocp.StandardCheckpointer()
  restored_params = checkpointer.restore(ckpt_path, target=abs_state)

  graph_def, _ = nnx.split(abs_gemma)
  gemma = nnx.merge(graph_def, restored_params)
  return gemma, mesh, model_config

In [ ]:
# This is a workaround. The checkpoints on Kaggle don't work with NNX. So, we
# load the model, save the checkpoint locally, and then reload the model
# (sharded).
if model_family == "gemma":
  params = params_lib.load_and_format_params(
      os.path.join(kaggle_ckpt_path, model_version)
  )
  gemma = gemma_lib.Transformer.from_params(params, version=model_version)
  checkpointer = ocp.StandardCheckpointer()
  _, state = nnx.split(gemma)
  checkpointer.save(os.path.join(INTERMEDIATE_CKPT_DIR, "state"), state)
  checkpointer.wait_until_finished()
  # Delete the intermediate model to save memory.
  del params
  del gemma
  del state
  gc.collect()
  # Base model
  base_model, mesh, model_config = get_base_model(
      ckpt_path=os.path.join(INTERMEDIATE_CKPT_DIR, "state"),
      model_version=model_version,
  )

In [ ]:
if model_family == "qwen3":
  mesh = jax.make_mesh(*MESH)
  model_version = "0.6b"  # Change to "14b" for Qwen-3 14B
  MODEL_CONFIG = {
      "0.6b": model.ModelConfig.qwen3_0_6b,
      "14b": model.ModelConfig.qwen3_14b,
  }
  kaggle_ckpt_path = kagglehub.model_download(
      f"{model_path[model_family]}{model_version}"
  )
  model_config = MODEL_CONFIG[model_version]()
  # Base model
  base_model = params.create_model_from_safe_tensors(
      kaggle_ckpt_path, config, mesh
  )

In [ ]:
nnx.display(base_model)

### Initialize Tokenizer

In [ ]:
dotenv.load_dotenv()
if model_family == "gemma":
  tokenizer = tokenizer_lib.Tokenizer(
      tokenizer_path=os.path.join(kaggle_ckpt_path, "tokenizer.model")
  )
elif model_family == "qwen3":
  tokenizer = tokenizer_lib.Tokenizer(
      tokenizer_type="huggingface",
      tokenizer_path=kaggle_ckpt_path,
      add_bos=True,
      add_eos=True,
      hf_access_token=os.environ.get("HF_TOKEN"),
  )

## Prompt the model

Let's see how the original model performs on the English-French translation task.

In [ ]:
sampler = sampler_lib.Sampler(
    transformer=base_model,
    tokenizer=tokenizer if model_family == "gemma" else tokenizer.tokenizer,
    cache_config=sampler_lib.CacheConfig(
        cache_size=256,
        num_layers=model_config.num_layers,
        num_kv_heads=model_config.num_kv_heads,
        head_dim=model_config.head_dim,
    ),
)

input_batch = [
    "Translate this into French:\nHello, my name is Morgane.\n",
    "Translate this into French:\nThis dish is delicious!\n",
    "Translate this into French:\nI am a student.\n",
    "Translate this into French:\nHow's the weather today?\n",
]

out_data = sampler(
    input_strings=input_batch,
    max_generation_steps=10,  # The number of steps performed when generating a response.
)

for input_string, out_string in zip(input_batch, out_data.text):
  print(f"----------------------")
  print(f"Prompt:\n{input_string}")
  print(f"Output:\n{out_string}")

## Apply LoRA/QLoRA to the base model

In [ ]:
def get_lora_model(base_model, mesh, quantize=False):
  if quantize:
    lora_provider = qwix.LoraProvider(
        module_path=".*q_einsum|.*kv_einsum|.*gate_proj|.*down_proj|.*up_proj",
        rank=RANK,
        alpha=ALPHA,
        weight_qtype="nf4",
        tile_size=256,
    )
  else:
    lora_provider = qwix.LoraProvider(
        module_path=".*q_einsum|.*kv_einsum|.*gate_proj|.*down_proj|.*up_proj",
        rank=RANK,
        alpha=ALPHA,
    )

  model_input = base_model.get_model_input()
  lora_model = qwix.apply_lora_to_model(
      base_model, lora_provider, **model_input
  )

  with mesh:
    state = nnx.state(lora_model)
    pspecs = nnx.get_partition_spec(state)
    sharded_state = jax.lax.with_sharding_constraint(state, pspecs)
    nnx.update(lora_model, sharded_state)

  return lora_model

In [ ]:
# LoRA model
lora_model = get_lora_model(base_model, mesh=mesh)
nnx.display(lora_model)

In [ ]:
# QLoRA model
qlora_model = get_lora_model(base_model, mesh=mesh, quantize=True)
nnx.display(qlora_model)

## Load Datasets for SFT Training

In [ ]:
# Loads the training and validation datasets
train_ds, validation_ds = data_lib.create_datasets(
    dataset_name='mtnt/en-fr',
    # Uncomment the line below to use a Hugging Face dataset.
    # Note that this requires upgrading the 'datasets' package and restarting
    # the Colab runtime if you are using it.
    # dataset_name='Helsinki-NLP/opus-100',
    global_batch_size=BATCH_SIZE,
    max_target_length=MAX_TARGET_LENGTH,
    num_train_epochs=NUM_EPOCHS,
    tokenizer=tokenizer,
)


def gen_model_input_fn(x: peft_trainer.TrainingInput):
  pad_mask = x.input_tokens != tokenizer.pad_id()
  positions = utils.build_positions_from_mask(pad_mask)
  attention_mask = utils.make_causal_attn_mask(pad_mask)
  return {
      'input_tokens': x.input_tokens,
      'input_mask': x.input_mask,
      'positions': positions,
      'attention_mask': attention_mask,
  }

## SFT Training

### Training with full weights

In [ ]:
logging_option = metrics_logger.MetricsLoggerOptions(
    log_dir="/tmp/tensorboard/full", flush_every_n_steps=20
)
training_config = peft_trainer.TrainingConfig(
    eval_every_n_steps=EVAL_EVERY_N_STEPS,
    max_steps=MAX_STEPS,
    metrics_logging_options=logging_option,
)
trainer = peft_trainer.PeftTrainer(
    base_model, optax.adamw(1e-5), training_config
)
trainer = trainer.with_gen_model_input_fn(gen_model_input_fn)

with jax.profiler.trace(os.path.join(PROFILING_DIR, "full_training")):
  with mesh:
    trainer.train(train_ds, validation_ds)

### Training with LoRA/QLoRA

In [ ]:
# Since LoRA model is sharing backbone with base model,
# restart Colab runtime so base model is loaded as pre-trained.

training_config = peft_trainer.TrainingConfig(
    eval_every_n_steps=EVAL_EVERY_N_STEPS,
    max_steps=MAX_STEPS,
    checkpoint_root_directory=CKPT_DIR,
)
lora_trainer = peft_trainer.PeftTrainer(
    lora_model, optax.adamw(1e-3), training_config
).with_gen_model_input_fn(gen_model_input_fn)

with jax.profiler.trace(os.path.join(PROFILING_DIR, "peft with LoRA")):
  with mesh:
    lora_trainer.train(train_ds, validation_ds)

In [ ]:
training_config = peft_trainer.TrainingConfig(
    eval_every_n_steps=EVAL_EVERY_N_STEPS,
    max_steps=MAX_STEPS,
    checkpoint_root_directory=CKPT_DIR,
)
qlora_trainer = peft_trainer.PeftTrainer(
    qlora_model, optax.adamw(1e-3), training_config
).with_gen_model_input_fn(gen_model_input_fn)

with jax.profiler.trace(os.path.join(PROFILING_DIR, "peft with QLoRA")):
  with mesh:
    qlora_trainer.train(train_ds, validation_ds)

### Compare profile results of different training

| Setup        | Train Step Time | Peak Memory Usage |
|--------------|----------------:|------------------:|
| Full weights |        ~1.22 s  |         43.26 GiB |
| QLoRA        |        ~1.19 s  |         28.14 GiB |

## Generate with the LoRA/QLoRA model

The QLoRA model still cannot do English-to-French translation properly since we
only trained for 100 steps. If you train it for longer, you will see better
results.

In [ ]:
sampler = sampler_lib.Sampler(
    transformer=lora_model,
    tokenizer=tokenizer if model_family == "gemma" else tokenizer.tokenizer,
    cache_config=sampler_lib.CacheConfig(
        cache_size=256,
        num_layers=model_config.num_layers,
        num_kv_heads=model_config.num_kv_heads,
        head_dim=model_config.head_dim,
    ),
)

input_batch = [
    "Translate this into French:\nHello, my name is Morgane.\n",
    "Translate this into French:\nThis dish is delicious!\n",
    "Translate this into French:\nI am a student.\n",
    "Translate this into French:\nHow's the weather today?\n",
]

out_data = sampler(
    input_strings=input_batch,
    max_generation_steps=10,  # The number of steps performed when generating a response.
)

for input_string, out_string in zip(input_batch, out_data.text):
  print(f"----------------------")
  print(f"Prompt:\n{input_string}")
  print(f"Output:\n{out_string}")